# AquaMark Theory and System Notes

This notebook aligns the **implemented AquaMark codebase** with the **project paper** in `AM_Documentation.pdf`.

- Paper focus: SVM-based predictive integrity detection.
- Runtime focus: PyTorch encoder/decoder watermark inference behind an Express API.


## 1. End-to-End Runtime Flow (Current Project)

1. User uploads image in `frontend/src/pages/EmbedPage.jsx` or `VerifyPage.jsx`.
2. Frontend calls Express endpoints in `backend/routes/*.js`.
3. Backend spawns `python backend/ai_model/predict.py` via `backend/helpers/runPython.js`.
4. Python module performs `embed` or `verify` inference.
5. JSON + previews are returned to frontend for visualization and download.


## 2. Watermark Embedding Model

Let:
- `I` be input image tensor in `[0,1]`.
- `b` be 64-bit watermark payload generated from text.
- `s` be embedding strength.
- `E(·)` be the encoder network.

Embedding output is represented as:

`I_w = E(I, b, s)`

The model also estimates residual `R = I_w - I`, constrained to remain low-amplitude so visual quality stays high (tracked by PSNR/SSIM).


## 3. Verification Model

The decoder `D(·)` predicts:
- bit probabilities `p_bits`,
- integrity probabilities `p_int = [p_absent, p_safe, p_corrupted]`.

Recovered bits are thresholded at `0.5`, then compared with expected bits from watermark text.

Bit Error Rate:

`BER = (1/N) * sum_i |b_i - bhat_i|`

Final status logic in `backend/ai_model/inference/verify.py` combines BER thresholds and classifier confidence into one of:
- `SAFE`
- `CORRUPTED`
- `ABSENT`


## 4. Key API Contract

Current backend (`backend/server.js`) exposes:
- `POST /api/embed-watermark`
- `POST /api/verify-watermark`
- `GET /api/model-status`
- `GET /api/download/:jobId`

The UI consumes these routes through `frontend/src/services/api.js`.


## 5. Mapping to AM_Documentation.pdf

`AM_Documentation.pdf` describes an SVM-based pipeline:
- watermark embedding `I_w = I + alpha W`,
- attack simulation `I_tilde_w = A(I_w; theta)`,
- statistical feature vector extraction,
- RBF-SVM integrity decision.

Reported paper performance: approximately `92-95%` accuracy on CIFAR-10 based experiments.

Current code preserves the same system objective (watermark integrity prediction) but uses a learned neural decoder instead of hand-crafted features + SVM at runtime.


## 6. Practical Reliability Notes

- If `checkpoint_best.pt` is missing, inference still runs but acts as untrained baseline.
- Frontend banners warn users when `model_status.trained` is false.
- For trustworthy results, use trained checkpoints and validate on attacked examples.


## 7. Documentation Contract

Keep both docs synchronized:
- `README.md` for setup, API, and operations.
- `theory.ipynb` for equations, architecture logic, and paper-to-code mapping.
